# `conftest.py` — Shared Pytest Fixtures

## Purpose

Provides **reusable pytest fixtures** that are automatically discovered and injected into every
test module in the `tests/` folder. Centralising fixture definitions here avoids duplication and
keeps each individual test file focused purely on assertions.

---

## Fixtures at a Glance

| Fixture | Returns | Description |
|---------|---------|-------------|
| `sample_df` | `pd.DataFrame` | 200-row synthetic HR dataset matching the real schema; ~24% attrition rate |
| `sample_csv_path` | `Path` | Writes `sample_df` to a temporary CSV file; path returned for loader tests |
| `preprocessed_data` | `tuple` | `(X_train, X_test, y_train, y_test)` produced by `DataPreprocessor.run_preprocessing_pipeline()` |

---

## How to Run Tests

```bash
# All tests with coverage
pytest tests/ -v --cov=src --cov-report=term-missing

# HTML coverage report
pytest tests/ -v --cov=src --cov-report=html
```

---

## Design Notes

- `sample_df` uses `np.random.seed(42)` for deterministic, reproducible data across all runs.
- Class imbalance (~24% `left=1`) mirrors the real dataset distribution so preprocessing tests
  reflect realistic conditions.
- `preprocessed_data` exercises the full `DataPreprocessor` pipeline, giving modeling and
  retention tests a ready-to-use train/test split without repeating setup logic.


---

### `sample_df` Fixture

**Purpose:** Generates a deterministic 200-row `pd.DataFrame` that exactly mirrors the schema
of `HR_comma_sep.csv`. Used as the base input for all unit tests that need an in-memory dataset.

**What it produces — step by step:**
1. Sets `np.random.seed(42)` so every run produces the same values.
2. Creates a `left` column with exactly 48 rows set to `1` (~24% attrition) to simulate
   realistic class imbalance.
3. Populates all 10 HR columns with random values within valid business ranges
   (e.g., `satisfaction_level` in [0.09, 1.0], `number_project` in [2, 7]).
4. Returns the assembled `DataFrame` — no side effects, no file I/O.

**Returns:** `pd.DataFrame` — 200 rows × 10 columns.


---

### `sample_csv_path` Fixture

**Purpose:** Persists `sample_df` to a temporary file on disk so that file-loading tests
(`DataQualityChecker.load_data()`) can be tested end-to-end with a real file path.

**What it produces — step by step:**
1. Receives `tmp_path` (pytest's built-in temporary directory) and `sample_df`.
2. Writes the DataFrame to `<tmp_path>/test_hr.csv` using `to_csv(index=False)`.
3. Returns the `Path` object pointing to the written file.

**Returns:** `Path` — absolute path to the temporary CSV file.

> **Note:** The temporary directory is automatically cleaned up by pytest after each test session.


---

### `preprocessed_data` Fixture

**Purpose:** Runs the full `DataPreprocessor` pipeline on `sample_df` and returns the four
train/test arrays required by `ModelTrainer`, `ModelEvaluator`, and `RetentionAdvisor` tests.

**What it produces — step by step:**
1. Imports `DataPreprocessor` from `src.preprocessing`.
2. Instantiates it with `sample_df`.
3. Calls `run_preprocessing_pipeline()` which: separates features/target → encodes categoricals
   → stratified split → applies SMOTE to training set.
4. Returns the resulting `(X_train, X_test, y_train, y_test)` tuple.

**Returns:** `tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]`


In [ ]:
"""
Shared pytest fixtures for all test modules.

Run tests from the project root:
    pytest tests/ -v --cov=src --cov-report=term-missing
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

# Ensure src/ is importable when running pytest from project root
PROJECT_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))


@pytest.fixture
def sample_df() -> pd.DataFrame:
    """Minimal DataFrame matching the HR dataset schema (200 rows)."""
    np.random.seed(42)
    n = 200
    # Force class imbalance: ~24% leave (realistic)
    left = np.zeros(n, dtype=int)
    left_indices = np.random.choice(n, size=48, replace=False)
    left[left_indices] = 1

    return pd.DataFrame(
        {
            "satisfaction_level": np.random.uniform(0.09, 1.0, n).round(2),
            "last_evaluation": np.random.uniform(0.36, 1.0, n).round(2),
            "number_project": np.random.randint(2, 8, n),
            "average_montly_hours": np.random.randint(96, 310, n),
            "time_spend_company": np.random.randint(2, 10, n),
            "Work_accident": np.random.randint(0, 2, n),
            "left": left,
            "promotion_last_5years": np.random.randint(0, 2, n),
            "sales": np.random.choice(
                ["sales", "technical", "hr", "IT", "support", "management"], n
            ),
            "salary": np.random.choice(["low", "medium", "high"], n),
        }
    )


@pytest.fixture
def sample_csv_path(tmp_path: Path, sample_df: pd.DataFrame) -> Path:
    """Write sample_df to a temporary CSV file and return its path."""
    csv_path = tmp_path / "test_hr.csv"
    sample_df.to_csv(csv_path, index=False)
    return csv_path


@pytest.fixture
def preprocessed_data(sample_df: pd.DataFrame):
    """Return (X_train, X_test, y_train, y_test) from the sample DataFrame."""
    from src.preprocessing.data_preprocessor import DataPreprocessor

    preprocessor = DataPreprocessor(sample_df)
    return preprocessor.run_preprocessing_pipeline()
